# Does the prefix signal survive holding out whole prefix families?

Registered test (D1-D3 committed before the run; see README). On the 2185 stationary census rules (43 families = first 6 letters), 8-fold cross-validation either by random rules or by whole families. Models: additive ridge A, additive plus pairwise A2, 8-letter lookup P8. Score: pooled out-of-fold strict R^2 against the training-fold mean.

**Assumptions.** Same stationarity filter as the other experiments; ridge penalty fixed at 1.0.

In [1]:
import json
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
d = json.loads((root / "experiments/archive/2026-10-kappa-prefix-family-holdout/results/holdout.json").read_text())
print("input:", {k: v[:12] for k, v in d["input_sha256"].items()}, "rules", d["n"], "families", d["n_families"])
for m in ("A", "A2", "P8"):
    r, g = d["r2"][f"random/{m}"], d["r2"][f"grouped/{m}"]
    print(f"{m:3s} random CV {r:+.3f}   family-held-out CV {g:+.3f}")
print(d["verdicts"])
assert list(d["verdicts"].values()) == [True, False, False]  # D1 held, D2 and D3 failed, as in the README
assert all(d["r2"][f"grouped/{m}"] < 0 for m in ("A", "A2", "P8"))
assert abs(d["r2"]["random/P8"] - 0.591) < 2e-3

input: {'probe_summary.csv': 'c76c7927185b'} rules 2185 families 43
A   random CV +0.109   family-held-out CV -0.162
A2  random CV +0.300   family-held-out CV -0.690
P8  random CV +0.591   family-held-out CV -0.162
{'D1 P8 random >= 0.45 and grouped < 0.30': True, 'D2 A2 grouped within 0.10 of random': False, 'D3 A2 grouped > P8 grouped': False}


## What this shows and does not show
- Holding out whole 6-letter prefix families, every model falls below the training mean (lookup 0.59 to -0.16; pairwise 0.30 to -0.69). The predictive signal is family membership, not a letter-level rule that transfers to unseen families.
- That answers the additive experiment's open question: the lookup's advantage over additive and pairwise models is memorisation of family means.
- **Not shown:** that no structure exists within families. Only 43 uneven families, about five held out per fold; negative R^2 largely reflects held-out family means differing from the training mean. No family random-effect model or other family granularity was tried.

## Next
A family-level random effect plus within-family letter structure, and holding out families at 4 and 8 letters.